# 🛠️ Desenvolvimento de IA para Análise Preditiva — Estimativa de RUL (SENAI / SCTEC)

**Desafio Escolhido — Opção B:** Pipeline preditivo desenvolvido a partir da base pública **Microsoft Azure Predictive Maintenance** (Kaggle / Azure AI Gallery), com foco na regressão da **Vida Útil Restante (RUL - Remaining Useful Life)** em horas de motores industriais.

---
## Fase 1: Preparação, Engenharia de Atributos e Serialização do Modelo (Pipeline Offline)
**Justificativa Técnica:**
- Os sensores industriais operam em alta frequência (1h), apresentando ruído estocástico. Aplicamos reamostragem temporal de **6 horas** (`resample('6h').mean()`) para estabilizar o sinal físico.
- Cruzamos as 5 tabelas relacionais (`PdM_telemetry`, `PdM_machines`, `PdM_failures`, `PdM_errors`, `PdM_maint`) usando junções temporais assíncronas (`pd.merge_asof`) para evitar vazamento de dados (*data leakage*).
- Construímos um `Pipeline` completo do Scikit-Learn integrando `SimpleImputer` (tratamento de nulos), `StandardScaler` (escalonamento numérico), `OneHotEncoder` (codificação categórica da família do motor `model1` a `model4`) e `RandomForestRegressor`.
- Os dados são divididos estritamente em **Treino (70%)**, **Validação (15%)** e **Teste (15%)**.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona a raiz do projeto ao sys.path usando caminho relativo (Item 5.1)
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.services.ml_service import (
    calculate_rul_from_azure_data,
    split_train_val_test,
    train_model,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET
)
print("Ambiente configurado com sucesso!")

### 1.1 Carregamento Relativo e Inspeção das 5 Tabelas da Azure (`data/`)

In [ ]:
data_dir = PROJECT_ROOT / "data"
df_machines = pd.read_csv(data_dir / "PdM_machines.csv")
df_failures = pd.read_csv(data_dir / "PdM_failures.csv")
df_errors = pd.read_csv(data_dir / "PdM_errors.csv")
df_maint = pd.read_csv(data_dir / "PdM_maint.csv")

print(f"Máquinas monitoradas: {len(df_machines)}")
print(f"Registros de falhas: {len(df_failures)}")
print(f"Registros de erros: {len(df_errors)}")
print(f"Registros de manutenções: {len(df_maint)}")
display(df_machines.head())

### 1.2 Consolidação Vetorizada do RUL e Divisão Treino (70%) / Validação (15%) / Teste (15%)

In [ ]:
df_rul = calculate_rul_from_azure_data(sample_machines=35)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(df_rul)

print(f"Shape consolidado: {df_rul.shape}")
print(f"Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}")
display(df_rul.describe())

### 1.3 Análise de Correlação entre Sensores, Histórico Operacional e RUL

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_rul[NUMERICAL_FEATURES + [TARGET]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlação - Telemetria Numérica e RUL (horas)")
plt.tight_layout()
plt.show()

### 1.4 Treinamento do Pipeline, Registro no MLflow e Serialização (`models/model.joblib`)

In [ ]:
metrics = train_model(n_estimators=150, max_depth=16, min_samples_leaf=2)
print("Relatório Oficial de Métricas (Validação e Teste):", metrics)

---
## Fases 2, 3, 4 e 5: Arquitetura Modular da API FastAPI e Ciclo de Vida
- **Fase 2 (Arquitetura em Camadas):** Implementada em `src/schemas/prediction.py`, `src/services/ml_service.py`, `src/routers/health.py`, `src/routers/predict.py` e `src/main.py`.
- **Fase 3 (Lifespan):** O artefato `models/model.joblib` é carregado uma única vez na memória RAM via `@asynccontextmanager` (`lifespan`) na inicialização do FastAPI (`src/main.py`).
- **Fase 4 (Validação Pydantic):** Restrições físicas estritas (`gt`, `ge`, `le`, `pattern`) configuradas com retorno automático de HTTP `422 Unprocessable Entity`.
- **Fase 5 (Endpoints e Tratamento HTTP):** Disponibilizados `GET /health` e `POST /predict` (e `/v1/predict`), além do painel interativo `GET /simulador`.